# AE2 – Manejo de archivos CSV, JSON y TXT

Kevin Daniel Sepúlveda Giraldo · Analítica de Datos · PREICA2602B020125

| Clase | Archivo | Uso |
|---|---|---|
| `ManejadorCSV` | `manejador_csv.py` | `csv.reader` / `csv.writer` |
| `ManejadorJSON` | `manejador_json.py` | `json.load` / `json.dump` |
| `GeneradorResumenTXT` | `generador_resumen_txt.py` | Genera `resumen.txt` |

## 1. Archivos de origen (Hugging Face)

In [ ]:
from pathlib import Path

URLS_ORIGEN = {
    "CSV": "https://huggingface.co/datasets/aptologia/barrios-mas-baratos-para-arrendar-colombia-2026/blob/main/data/barrios_mas_baratos_para_arrendar_colombia_2026_datos.csv",
    "JSON": "https://huggingface.co/datasets/Vault-of-History/colombian-numismatic-collection/blob/main/colombian-coins.json",
}

CSV_ORIGINAL = Path("datos/originales/barrios_mas_baratos_para_arrendar_colombia_2026_datos.csv")
JSON_ORIGINAL = Path("datos/originales/colombian-coins.json")

CSV_MODIFICADO = Path("datos/modificados/barrios_arriendo_modificado.csv")
JSON_MODIFICADO = Path("datos/modificados/monedas_colombia_modificado.json")

for ruta in (CSV_ORIGINAL, JSON_ORIGINAL):
    print(f"{ruta} -> existe: {ruta.exists()}")

## 2. Importar las clases

In [ ]:
from manejador_csv import ManejadorCSV
from manejador_json import ManejadorJSON
from generador_resumen_txt import GeneradorResumenTXT

manejador_csv = ManejadorCSV(CSV_ORIGINAL, URLS_ORIGEN["CSV"])
manejador_json = ManejadorJSON(JSON_ORIGINAL, URLS_ORIGEN["JSON"])

## 3. Leer los archivos

In [ ]:
filas_csv = manejador_csv.importar()
registros_json = manejador_json.importar()

print(f"CSV importado:  {filas_csv} filas | encabezados: {manejador_csv.encabezados}")
print(f"JSON importado: {registros_json} registros")

## 4. CSV: barrios más baratos para arrendar

**Antes:**

In [ ]:
for fila in manejador_csv.vista_previa(5):
    print(fila)

Cambios:

1. Agregar `canon_millones`.
2. Cambiar `Bogotá` por `Bogotá D.C.`.
3. Eliminar barrios con menos de 35 avisos (muy pocos anuncios para una mediana confiable).

In [ ]:
manejador_csv.agregar_columna(
    "canon_millones",
    lambda fila: round(float(fila["canon_cop_mediana"]) / 1_000_000, 2),
)
cambiados = manejador_csv.actualizar_valor("city", "Bogotá", "Bogotá D.C.")
eliminados = manejador_csv.eliminar_filas(
    lambda fila: int(fila["avisos"]) < 35,
    "barrios con menos de 35 avisos",
)

print(f"Filas con 'Bogotá' actualizadas: {cambiados}")
print(f"Filas eliminadas: {eliminados}")
print(f"Filas antes: {filas_csv} | Filas después: {len(manejador_csv.filas)}")

**Después:**

In [ ]:
for fila in manejador_csv.vista_previa(5):
    print(fila)

## 5. JSON: monedas colombianas

**Antes:**

In [ ]:
CAMPOS_A_MOSTRAR = ["_id", "year_of_issue", "denomination", "unit", "status", "photo"]

for registro in manejador_json.vista_previa(3, CAMPOS_A_MOSTRAR):
    print(registro)

Cambios:

1. Agregar `antiguedad_anios` (2026 − año de emisión).
2. Cambiar `No longer in circulation` por `Fuera de circulación`.
3. Eliminar `photo` (nombre de una imagen que no viene en el dataset).

In [ ]:
manejador_json.agregar_campo(
    "antiguedad_anios",
    lambda moneda: 2026 - moneda["year_of_issue"] if isinstance(moneda.get("year_of_issue"), int) else None,
)
actualizados = manejador_json.actualizar_valor("status", "No longer in circulation", "Fuera de circulación")
sin_foto = manejador_json.eliminar_campo("photo")

print(f"Registros con estado actualizado: {actualizados}")
print(f"Registros a los que se les quitó 'photo': {sin_foto}")

**Después:**

In [ ]:
CAMPOS_DESPUES = ["_id", "year_of_issue", "antiguedad_anios", "denomination", "unit", "status"]

for registro in manejador_json.vista_previa(3, CAMPOS_DESPUES):
    print(registro)

quedan_con_foto = sum("photo" in moneda for moneda in manejador_json.registros)
print(f"\nRegistros que todavía tienen el campo 'photo': {quedan_con_foto}")

## 6. Guardar y comprobar

Los archivos nuevos quedan en `datos/modificados/`; los originales no se tocan.

In [ ]:
ruta_csv = manejador_csv.guardar(CSV_MODIFICADO)
ruta_json = manejador_json.guardar(JSON_MODIFICADO)
print(f"CSV guardado en:  {ruta_csv}")
print(f"JSON guardado en: {ruta_json}")

verificacion_csv = ManejadorCSV(ruta_csv, URLS_ORIGEN["CSV"])
verificacion_json = ManejadorJSON(ruta_json, URLS_ORIGEN["JSON"])

print(f"\nCSV nuevo:  {verificacion_csv.importar()} filas | encabezados: {verificacion_csv.encabezados}")
print(f"¿Tiene la columna canon_millones? {'canon_millones' in verificacion_csv.encabezados}")

print(f"\nJSON nuevo: {verificacion_json.importar()} registros")
primera = verificacion_json.registros[0]
print(f"¿Tiene antiguedad_anios? {'antiguedad_anios' in primera} | "
      f"¿Todavía tiene photo? {'photo' in primera} | status: {primera.get('status')}")

print(f"\nLos originales siguen ahí: {CSV_ORIGINAL.exists()} y {JSON_ORIGINAL.exists()}")

## 7. Generar `resumen.txt`

In [ ]:
generador = GeneradorResumenTXT(
    titulo="Resumen AE2 - Manejo de archivos CSV y JSON",
    urls_origen=URLS_ORIGEN,
)
generador.agregar_informe(manejador_csv.informe())
generador.agregar_informe(manejador_json.informe())

ruta_resumen = generador.guardar("resumen.txt")
print(f"Resumen guardado en: {ruta_resumen}\n")
print(ruta_resumen.read_text(encoding="utf-8"))